In [1]:
import ast
from datasets import load_dataset

/Users/chuu/Documents/tutorials/Stravl-Data-Huggingface/notebooks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset('csv', data_files='../Stravl_Travel_Preference_Data.csv')

with open("../destination_ids.txt", "r") as f:
    destination_ids = f.readlines()
    destination_ids = [destination_id.strip() for destination_id in destination_ids]

In [3]:
destination_ids

['Butrint, Albania',
 'Tirana, Albania',
 'Algiers, Algeria',
 'Oran, Algeria',
 'Andorra la Vella, Andorra',
 'Jolly Harbour, Antigua and Barbuda',
 "St John's, Antigua and Barbuda",
 'Buenos Aires, Argentina',
 'Córdoba, Argentina',
 'Mar del Plata, Argentina',
 'Mendoza, Argentina',
 'Puerto Iguazu, Argentina',
 'Puerto Madryn, Argentina',
 'Salta, Argentina',
 'San Carlos de Bariloche, Argentina',
 'San Martin de los Andes, Argentina',
 'San Miguel de Tucuman, Argentina',
 'Ushuaia, Argentina',
 'Villa Gesell, Argentina',
 'Garni, Armenia',
 'Sevan, Armenia',
 'Yerevan, Armenia',
 'Adelaide, Australia',
 'Alice Springs, Australia',
 'Brisbane, Australia',
 'Broome, Australia',
 'Byron Bay, Australia',
 'Cairns, Australia',
 'Canberra, Australia',
 'Darwin, Australia',
 'Geelong, Australia',
 'Hobart, Australia',
 'Launceston, Australia',
 'Melbourne, Australia',
 'Newcastle, Australia',
 'Perth, Australia',
 'Sydney, Australia',
 'Bad Gastein, Austria',
 'Bregenz, Austria',
 'Graz,

In [4]:
dataset['train']

Dataset({
    features: ['id', 'form_a', 'form_b', 'form_c', 'form_f', 'form_g', 'form_h', 'form_i', 'form_j', 'form_k', 'form_r', 'form_rr', 'yes_swipes', 'no_swipes', 'maybe_swipes', 'Model', 'Retrieval', 'DynaMatch', 'Rating_0', 'Rating_1', 'Rating_2', 'Rating_3', 'Rating_4', 'Rating_5', 'Rating_6', 'Rating_7', 'Rating_8', 'Rating_9', 'Rec_0', 'Rec_1', 'Rec_2', 'Rec_3', 'Rec_4', 'Rec_5', 'Rec_6', 'Rec_7', 'Rec_8', 'Rec_9'],
    num_rows: 80301
})

In [5]:
SWIPE_COLS = ['yes_swipes', 'no_swipes', 'maybe_swipes']

def add_swipe_names(example):
    for col in SWIPE_COLS:
        indexes = example[col]
        if isinstance(indexes, str):
            indexes = ast.literal_eval(indexes)
        example[f'{col}_names'] = [destination_ids[i] for i in indexes]
    return example

dataset = dataset.map(add_swipe_names)

# Place each *_names column immediately after its source column
cols = dataset['train'].column_names
ordered = []
for c in cols:
    if c.endswith('_names'):
        continue
    ordered.append(c)
    if c in SWIPE_COLS:
        ordered.append(f'{c}_names')
dataset['train'] = dataset['train'].select_columns(ordered)

In [6]:
# Map form-response codes to their human-readable labels (from README.md).
# form_k is intentionally omitted (undocumented in the README).
FORM_VALUE_MAPS = {
    'form_a': {'0': '0-19', '1': '20-39', '2': '40-59', '3': '60+'},
    'form_b': {'0': '$0-$49', '1': '$50-$99', '2': '$100-$249', '3': '$300+'},
    'form_c': {'0': 'Winter', '1': 'Spring', '2': 'Summer', '3': 'Fall'},
    'form_f': {'0': 'Beach', '1': 'Adventure', '2': 'Nature', '3': 'Culture',
               '4': 'Nightlife', '5': 'History', '6': 'Shopping', '7': 'Cuisine'},
    'form_g': {'0': 'Urban', '1': 'Rural', '2': 'Sea', '3': 'Mountain',
               '4': 'Lake', '5': 'Desert', '6': 'Plains', '7': 'Jungle'},
    'form_h': {'0': 'Chill & Relaxed', '1': 'Balanced', '2': 'Active'},
    'form_i': {'0': 'Very Safety Conscious', '1': 'Balanced', '2': 'Ready for Anything'},
    'form_j': {'0': 'Off the Beaten Path', '1': 'Classic Spot', '2': 'Mainstream & Trendy'},
    'form_r': {'0': 'Anywhere', '1': 'Specific Regions'},
    'form_rr': {'e': 'Europe', 'n': 'N. America', 'c': 'Caribbean', 'a': 'Asia',
                's': 'S. America', 'm': 'Mid. East', 'f': 'Africa', 'o': 'Oceania'},
}

# Descriptive output column name for each form (suffix reflects the README question).
FORM_NAME_COLS = {
    'form_a': 'form_a_age_ranges',
    'form_b': 'form_b_trip_budget',
    'form_c': 'form_c_travel_season',
    'form_f': 'form_f_experiences',
    'form_g': 'form_g_scenery',
    'form_h': 'form_h_activity_level',
    'form_i': 'form_i_safety_conscious',
    'form_j': 'form_j_destination_popularity',
    'form_r': 'form_r_destination_choice',
    'form_rr': 'form_rr_specific_regions',
}

def add_form_names(example):
    for col, mapping in FORM_VALUE_MAPS.items():
        codes = example[col]
        if isinstance(codes, str):
            codes = ast.literal_eval(codes) if codes.strip() else []
        if not codes:
            codes = []
        example[FORM_NAME_COLS[col]] = [mapping.get(c, c) for c in codes]
    return example

dataset = dataset.map(add_form_names)

# Place each descriptive name column immediately after its source column
ordered = []
generated = set(FORM_NAME_COLS.values())
for c in dataset['train'].column_names:
    if c in generated:
        continue
    ordered.append(c)
    if c in FORM_NAME_COLS:
        ordered.append(FORM_NAME_COLS[c])
dataset['train'] = dataset['train'].select_columns(ordered)

In [7]:
dataset['train']

Dataset({
    features: ['id', 'form_a', 'form_a_age_ranges', 'form_b', 'form_b_trip_budget', 'form_c', 'form_c_travel_season', 'form_f', 'form_f_experiences', 'form_g', 'form_g_scenery', 'form_h', 'form_h_activity_level', 'form_i', 'form_i_safety_conscious', 'form_j', 'form_j_destination_popularity', 'form_k', 'form_r', 'form_r_destination_choice', 'form_rr', 'form_rr_specific_regions', 'yes_swipes', 'yes_swipes_names', 'no_swipes', 'no_swipes_names', 'maybe_swipes', 'maybe_swipes_names', 'Model', 'Retrieval', 'DynaMatch', 'Rating_0', 'Rating_1', 'Rating_2', 'Rating_3', 'Rating_4', 'Rating_5', 'Rating_6', 'Rating_7', 'Rating_8', 'Rating_9', 'Rec_0', 'Rec_1', 'Rec_2', 'Rec_3', 'Rec_4', 'Rec_5', 'Rec_6', 'Rec_7', 'Rec_8', 'Rec_9'],
    num_rows: 80301
})

In [8]:
dataset['train'][1]

{'id': '89b7022ab8f011ec9a19fefb5d5b7ead',
 'form_a': "['0', '1']",
 'form_a_age_ranges': ['0-19', '20-39'],
 'form_b': "['3']",
 'form_b_trip_budget': ['$300+'],
 'form_c': "['2']",
 'form_c_travel_season': ['Summer'],
 'form_f': "['0', '3', '4', '5', '6', '7']",
 'form_f_experiences': ['Beach',
  'Culture',
  'Nightlife',
  'History',
  'Shopping',
  'Cuisine'],
 'form_g': "['0', '1', '2', '4']",
 'form_g_scenery': ['Urban', 'Rural', 'Sea', 'Lake'],
 'form_h': "['1']",
 'form_h_activity_level': ['Balanced'],
 'form_i': "['1']",
 'form_i_safety_conscious': ['Balanced'],
 'form_j': "['1']",
 'form_j_destination_popularity': ['Classic Spot'],
 'form_k': "['1']",
 'form_r': None,
 'form_r_destination_choice': [],
 'form_rr': None,
 'form_rr_specific_regions': [],
 'yes_swipes': '[246, 579]',
 'yes_swipes_names': ['Luxor, Egypt', 'Utrecht, Netherlands'],
 'no_swipes': '[126, 945, 865, 330, 1237, 1264, 824, 87]',
 'no_swipes_names': ['Montreal, Canada',
  'Palm Springs, California, United 

In [9]:
dataset.push_to_hub("chuuhtetnaing/stravl-travel-preference-dataset", private=True, token="")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  6.79ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 43.8MB / 43.8MB,   ???B/s  
New Data Upload: |          |  0.00B /  0.00B,   ???B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.74s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/chuuhtetnaing/stravl-travel-preference-datasett/commit/e02703efcebae1de9e1908f77c7b82af219bf756', commit_message='Upload dataset', commit_description='', oid='e02703efcebae1de9e1908f77c7b82af219bf756', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/chuuhtetnaing/stravl-travel-preference-datasett', endpoint='https://huggingface.co', repo_type='dataset', repo_id='chuuhtetnaing/stravl-travel-preference-datasett'), pr_revision=None, pr_num=None)